# E10a · Redes neuronales entrenadas desde cero
**Parte de:** E8b (stacking con bagging de 5 semillas, 5 modelos + LightGBM; CV 0,8179).

**Hipótesis.** Todos los modelos anteriores ven el texto como una bolsa de n-gramas, sin orden. Una red que lee la secuencia (BiGRU, CNN de caracteres) podría captar estructura (quién insulta a quién) y aportar diversidad al stacking.

**Cumplimiento de las reglas.** Las redes se entrenan desde cero con PyTorch: embeddings aprendidos y ningún vector preentrenado. Los vocabularios se ajustan solo con la partición de entrenamiento de cada fold, y el test nunca entra en un ajuste.

**Modelos** (código en `ext_neuronal.py`):

| Modelo | Arquitectura |
|---|---|
| NN-BolsaEmb | Bolsa de embeddings de palabras (estilo fastText) |
| NN-BiGRU | GRU bidireccional sobre palabras |
| NN-CharCNN | CNN sobre caracteres |
| NN-Mix | Palabras + caracteres |
| NN-BolsaEmb-lex / NN-BiGRU-lex | La misma red + las 38 variables de léxico H1-H10 en la capa densa |

Entrenamiento: 3 sigmoides (BCE con `pos_weight` balanceado), AdamW, hasta 40 épocas y *early stopping* (paciencia 5) sobre un 15 % interno del train del fold. Se promedian 3 inicializaciones.

Por el costo de entrenamiento, este notebook **lee los resultados de la caché** (`oof/base/NN-*`). Para recalcularlos, borra esos archivos y llama a `tb.oof(d, nombre, s)` después de `import ext_neuronal`.

### Glosario de esta prueba
| Término | Significado |
|---|---|
| **HS / TR / AG** | Las 3 etiquetas: discurso de odio / odio dirigido a una persona concreta / agresividad |
| **F1 medio** | F1 de la clase 1 de cada etiqueta, promediado entre HS, TR y AG (métrica principal) |
| **CV repetida** | Validación cruzada de 5 particiones repetida con 3 semillas (42, 43, 44); se reporta media ± desviación |
| **OOF** | *Out-of-fold*: predicción de cada fila del train hecha por un modelo que no la vio al entrenar |
| **Stacking** | Un metamodelo (regresión logística) aprende a combinar las probabilidades OOF de los modelos base |
| **BiGRU** | Red recurrente bidireccional: lee la secuencia de palabras en ambos sentidos |
| **CNN de caracteres** | Red convolucional que detecta patrones de letras |
| **Embedding aprendido** | Vector por palabra que la red aprende durante el entrenamiento (no preentrenado) |
| **Early stopping** | Detener el entrenamiento cuando deja de mejorar en una validación interna |

El glosario completo está en `RESUMEN.md`.

In [1]:
import os, sys, platform
sys.path.insert(0, os.path.abspath('.'))          # el notebook vive en experimentos/, junto a toxic_base.py
import numpy as np, pandas as pd, sklearn, scipy
import toxic_base as tb
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 30); pd.set_option('display.max_colwidth', 120)
print('Python', platform.python_version(), '| numpy', np.__version__, '| pandas', pd.__version__,
      '| scikit-learn', sklearn.__version__, '| scipy', scipy.__version__)
print('Semillas de la CV repetida:', tb.SEEDS, '| particiones:', tb.N_FOLDS)
d = tb.Datos()
print('Datos en', d.data_dir, '| train', d.train.shape, '| test', d.test.shape)

Python 3.12.3 | numpy 2.5.4 | pandas 3.0.6 | scikit-learn 1.9.1 | scipy 1.18.1
Semillas de la CV repetida: (42, 43, 44) | particiones: 5


Datos en /home/jc/repositories/natural-language-processing-workshops/workshops/Kaggle-tweet-toxic/tweet-toxic-2026-puj | train (1731, 5) | test (1154, 2)


In [2]:
import ext_neuronal
R = {'LR (referencia)': tb.evaluar(d, {s: tb.oof(d, 'LR', s) for s in tb.SEEDS})}
for b in ['NN-BolsaEmb-lex', 'NN-BiGRU-lex', 'NN-BolsaEmb']:
    R[b] = tb.evaluar(d, {s: tb.oof(d, b, s) for s in tb.SEEDS})
R['NN-BiGRU (solo semilla 42)'] = tb.evaluar(d, {42: tb.oof(d, 'NN-BiGRU', 42)})
tb.tabla(R)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
LR (referencia),0.7924,0.8246,0.7899,0.8023,0.0020,0.7499,0.803 / 0.805 / 0.800
NN-BolsaEmb-lex,0.7698,0.8151,0.7722,0.7857,0.0036,0.7556,0.790 / 0.781 / 0.786
NN-BiGRU-lex,0.7588,0.8052,0.7534,0.7724,0.0022,0.7448,0.772 / 0.775 / 0.770
NN-BolsaEmb,0.7074,0.7706,0.7004,0.7261,0.0034,0.6842,0.731 / 0.723 / 0.725
NN-BiGRU (solo semilla 42),0.7047,0.7952,0.7153,0.7384,0.0000,0.6996,0.738


NN-CharCNN (0,690) y NN-Mix (0,713) se descartaron tras el primer fold de la semilla 42, por debajo del umbral de 0,77.

## Stacking

In [3]:
S5 = (42, 43, 44, 45, 46); B6 = tb.BASES_ORIGINALES + ['LGBM']
RS = {'E8b · B6': tb.evaluar(d, {s: tb.stacking_bagged(d, B6, S5, s, con_test=False)[0] for s in tb.SEEDS}),
      'B6 + NN-BolsaEmb-lex': tb.evaluar(d, {s: tb.stacking_bagged(d, B6 + ['NN-BolsaEmb-lex'], S5, s, con_test=False)[0] for s in tb.SEEDS})}
tb.tabla(RS)

,F1 HS,F1 TR,F1 AG,F1 medio,±,EMR,F1 medio por semilla
Configuración,,,,,,,
E8b · B6,0.8071,0.8405,0.8062,0.8179,0.0012,0.7785,0.819 / 0.818 / 0.816
B6 + NN-BolsaEmb-lex,0.8083,0.8385,0.8056,0.8175,0.0006,0.7780,0.818 / 0.818 / 0.817


## Conclusión: **no se adopta**
- Las redes puras quedan en 0,69-0,74: con 1.731 tweets no alcanzan a la LR con n-gramas (0,802).
- Solo las versiones con léxico pasan de 0,77, porque reciben las mismas señales hechas a mano que ya usan las otras bases. Por eso no aportan diversidad, y el stacking queda igual (0,8175 frente a 0,8179).
- Lectura para el informe: con tan pocos datos, el orden de las palabras no compensa la falta de vectores preentrenados (que las reglas prohíben).